# Composite Systems and Entanglement
Chapter 6 companion. Convention: |AB⟩=|q1 q0⟩, basis order 00,01,10,11. A=q1 and B=q0, even though the default circuit drawing places q0 above q1. Predictions are ideal. Stored statevectors are model data, not measurement records.


In [ ]:
%pip -q install "qiskit>=2,<3" "numpy>=2,<3" "matplotlib>=3.8,<4"


In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Operator
z=np.array([1,0],complex)
o=np.array([0,1],complex)
I=np.eye(2,dtype=complex)
H=np.array([[1,1],[1,-1]],complex)/np.sqrt(2)
X=np.array([[0,1],[1,0]],complex)
S=np.diag([1,1j])
plus=H@z
CX=np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]],complex)


## 1. Product states and ordering
Use an asymmetric state to test the mapping. A symmetric Bell state alone cannot detect a swapped convention.


In [ ]:
qc=QuantumCircuit(2)
qc.h(1) # A
qc.x(0) # B
v=Statevector.from_instruction(qc)
assert np.allclose(v.data,np.kron(plus,o))
assert np.allclose(v.probabilities(),[0,.5,0,.5])
assert np.allclose(v.probabilities([1]),[.5,.5]) # A marginal
assert np.allclose(v.probabilities([0]),[0,1]) # B marginal
print(qc.draw(output="text"))
print(v.probabilities_dict())


## 2. Pure-state factorization
For a normalized two-qubit pure column, det(C)=0 characterizes product states. A numerical tolerance is used only for floating-point comparisons. This is not a test on a histogram or an arbitrary mixed state.


In [ ]:
def determinant(v):
    a,b,c,d=np.asarray(v)
    return a*d-b*c
bell=np.array([1,0,0,1],complex)/np.sqrt(2)
assert np.isclose(determinant(np.kron(plus,plus)),0)
assert np.isclose(determinant(bell),.5)
for theta in [0,np.pi/8,np.pi/4,np.pi/2]:
    v=np.array([np.cos(theta),0,0,np.sin(theta)],complex)
    print(theta,determinant(v))


## 3. Local operators and Bell preparation
Check both tensor positions and the control-target ordering against independent matrices.


In [ ]:
qc=QuantumCircuit(2)
qc.h(1)
assert np.allclose(Operator(qc).data,np.kron(H,I))
qc_b=QuantumCircuit(2)
qc_b.h(0)
assert np.allclose(Operator(qc_b).data,np.kron(I,H))
cx=QuantumCircuit(2)
cx.cx(1,0)
assert np.allclose(Operator(cx).data,CX)
qc.cx(1,0)
assert np.allclose(Statevector.from_instruction(qc).data,bell)
assert np.allclose(CX@np.kron(H,I)@np.kron(z,z),bell)
assert np.allclose(CX@np.kron(plus,plus),np.kron(plus,plus))
assert np.allclose(np.kron(H,I)@bell,np.array([1,1,1,-1])/2)
print(qc.draw(output="text"))


## 4. Partial-measurement branches
Select a branch mathematically; the selector cannot force an outcome. For X/Y, rotate only the measured subsystem into the detector basis. The returned two-entry vector describes the unmeasured subsystem.


In [ ]:
def readout(basis):
    return {"Z":I,"X":H,"Y":H@S.conj().T}[basis]
def branch(v,which,outcome,basis="Z"):
    U=np.kron(readout(basis),I) if which=="A" else np.kron(I,readout(basis))
    w=(U@v).reshape(2,2)
    raw=w[outcome,:] if which=="A" else w[:,outcome]
    p=float(np.vdot(raw,raw).real)
    return p,None if p==0 else raw/np.sqrt(p)
psi=np.array([1,1j,1,0])/np.sqrt(3)
p,b=branch(psi,"A",0)
assert np.isclose(p,2/3)
assert np.allclose(b,[1/np.sqrt(2),1j/np.sqrt(2)])
p1,b1=branch(psi,"A",1)
assert np.isclose(p1,1/3)
assert np.allclose(b1,z)
assert np.isclose(p*abs(b[0])**2+p1*abs(b1[0])**2,2/3)
p_impossible,v_impossible=branch(np.kron(z,plus),"A",1)
assert p_impossible==0 and v_impossible is None
sv=Statevector(psi)
sv.seed(2026)
record,post=sv.measure([1]) # A=q1
p_selected,b_selected=branch(psi,"A",int(record))
expected=np.kron(z if record=="0" else o,b_selected)
assert Statevector(expected).equiv(post)
print("Random A outcome:",record,"branch probability:",p_selected)


## 5. Conditional versus local predictions
For the Bell state, measuring A in X gives conditional plus/minus on B. Averaging over the record leaves B uniform in any of the three Pauli bases.


In [ ]:
for alice_basis in ["Z","X","Y"]:
    for bob_basis in ["Z","X","Y"]:
        averaged=np.zeros(2)
        for outcome in [0,1]:
            p,b=branch(bell,"A",outcome,alice_basis)
            averaged += p*np.abs(readout(bob_basis)@b)**2
        assert np.allclose(averaged,[.5,.5])
for outcome,target in [(0,plus),(1,H@o)]:
    p,b=branch(bell,"A",outcome,"X")
    assert np.isclose(p,.5) and Statevector(b).equiv(Statevector(target))


## 6. Bell versus classical correlation
The randomized preparation averages probabilities, not amplitudes. Use fresh pairs for every shot. This comparison is not by itself the full CHSH protocol.


In [ ]:
def joint_probs(v,a="Z",b="Z"):
    return np.abs(np.kron(readout(a),readout(b))@v)**2
for basis in ["Z","X","Y"]:
    quantum=joint_probs(bell,basis,basis)
    classical=.5*joint_probs(np.kron(z,z),basis,basis)+.5*joint_probs(np.kron(o,o),basis,basis)
    print(basis,"Bell:",quantum,"random 00/11:",classical)
    if basis=="Z":
        assert np.allclose(quantum,classical)
    elif basis=="X":
        assert np.allclose(quantum,[.5,0,0,.5])
        assert np.allclose(classical,[.25]*4)
    else:
        assert np.allclose(quantum,[0,.5,.5,0])
rng=np.random.default_rng(2026)
print("200 fresh XX Bell counts:",rng.multinomial(200,joint_probs(bell,"X","X")))
print("200 fresh XX classical counts:",rng.multinomial(200,[.25]*4))


## Practice
1. Prepare a partially entangled state with Ry on A followed by CNOT. Verify its column and conditional branches.
2. Replace A=q1,B=q0 by the opposite mapping and explicitly permute the columns needed to compare results.
3. Calculate EZZ+EXX for the Bell state and random 00/11 ensemble; explain the trusted-measurement assumptions of the separable bound.

References: [Qiskit bit ordering](https://quantum.cloud.ibm.com/docs/en/guides/bit-ordering), [Statevector API](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.quantum_info.Statevector).
